
# 02 - Data Cleaning and Preprocessing

## Project: SmartCommerce – Brazilian E-Commerce Analytics

### Objective
To clean, validate, and preprocess the Brazilian E-Commerce dataset for reliable business analysis and machine learning.

### Tasks
1. Load raw datasets.
2. Identify missing values.
3. Detect duplicate records.
4. Correct data types.
5. Handle inconsistent values.
6. Detect and investigate outliers.
7. Validate cleaned datasets.
8. Save processed datasets.

### Tools
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn


In [1]:

# Step 1: Import Required Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Step 2: Configure Project Paths

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

# Create processed directory if necessary
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Raw Data Path:", RAW_DATA_PATH)
print("Processed Data Path:", PROCESSED_DATA_PATH)

print("\nRaw dataset folder exists:",
      RAW_DATA_PATH.exists())


Project Root: c:\Users\Rakesh bade\Desktop\SmartCommerce
Raw Data Path: c:\Users\Rakesh bade\Desktop\SmartCommerce\data\raw
Processed Data Path: c:\Users\Rakesh bade\Desktop\SmartCommerce\data\processed

Raw dataset folder exists: True


In [3]:

# Step 3: Load Raw Datasets

customers = pd.read_csv(
    RAW_DATA_PATH / "olist_customers_dataset.csv"
)

orders = pd.read_csv(
    RAW_DATA_PATH / "olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    RAW_DATA_PATH / "olist_order_items_dataset.csv"
)

payments = pd.read_csv(
    RAW_DATA_PATH / "olist_order_payments_dataset.csv"
)

reviews = pd.read_csv(
    RAW_DATA_PATH / "olist_order_reviews_dataset.csv"
)

products = pd.read_csv(
    RAW_DATA_PATH / "olist_products_dataset.csv"
)

sellers = pd.read_csv(
    RAW_DATA_PATH / "olist_sellers_dataset.csv"
)

geolocation = pd.read_csv(
    RAW_DATA_PATH / "olist_geolocation_dataset.csv"
)

category_translation = pd.read_csv(
    RAW_DATA_PATH / "product_category_name_translation.csv"
)

print("All 9 datasets loaded successfully!")


All 9 datasets loaded successfully!


In [4]:

# Step 4: Organize All Datasets

datasets = {
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation
}

print("Total datasets:", len(datasets))


Total datasets: 9


In [5]:

# Step 5: Dataset Dimensions

for name, df in datasets.items():
    print(f"\nDataset: {name}")
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])



Dataset: customers
Rows: 99441
Columns: 5

Dataset: orders
Rows: 99441
Columns: 8

Dataset: order_items
Rows: 112650
Columns: 7

Dataset: payments
Rows: 103886
Columns: 5

Dataset: reviews
Rows: 99224
Columns: 7

Dataset: products
Rows: 32951
Columns: 9

Dataset: sellers
Rows: 3095
Columns: 4

Dataset: geolocation
Rows: 1000163
Columns: 5

Dataset: category_translation
Rows: 71
Columns: 2


In [6]:

# Step 6: Missing Values Analysis

missing_summary = []

for name, df in datasets.items():

    for column in df.columns:

        missing_count = df[column].isnull().sum()

        missing_percentage = (
            missing_count / len(df) * 100
        )

        missing_summary.append({
            "Dataset": name,
            "Column": column,
            "Missing Values": missing_count,
            "Missing Percentage": round(
                missing_percentage, 2
            )
        })

missing_df = pd.DataFrame(missing_summary)

missing_df = missing_df[
    missing_df["Missing Values"] > 0
]

missing_df.sort_values(
    by="Missing Percentage",
    ascending=False
)


,Dataset,Column,Missing Values,Missing Percentage
28,reviews,review_comment_title,87656,88.34
29,reviews,review_comment_message,58247,58.70
11,orders,order_delivered_customer_date,2965,2.98
34,products,product_name_lenght,610,1.85
33,products,product_category_name,610,1.85
35,products,product_description_lenght,610,1.85
36,products,product_photos_qty,610,1.85
10,orders,order_delivered_carrier_date,1783,1.79
9,orders,order_approved_at,160,0.16
37,products,product_weight_g,2,0.01


In [7]:

# Step 7: Duplicate Records Analysis

duplicate_summary = []

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    duplicate_summary.append({
        "Dataset": name,
        "Total Rows": len(df),
        "Duplicate Rows": duplicate_count
    })

duplicate_df = pd.DataFrame(duplicate_summary)

duplicate_df


,Dataset,Total Rows,Duplicate Rows
0,customers,99441,0
1,orders,99441,0
2,order_items,112650,0
3,payments,103886,0
4,reviews,99224,0
5,products,32951,0
6,sellers,3095,0
7,geolocation,1000163,261831
8,category_translation,71,0


In [8]:

# Step 8: Data Types Analysis

for name, df in datasets.items():

    print("\nDataset:", name)

    print(df.dtypes)



Dataset: customers
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Dataset: orders
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

Dataset: order_items
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Dataset: payments
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

Dataset: reviews
revie

In [9]:

# Step 9: Investigate Geolocation Duplicates

print("Original Geolocation Shape:")
print(geolocation.shape)

print("\nDuplicate Rows:")
print(geolocation.duplicated().sum())

print("\nDuplicate Percentage:")
print(
    round(
        geolocation.duplicated().mean() * 100, 2
    ), "%"
)

print("\nSample Duplicate Records:")

geolocation[
    geolocation.duplicated(keep=False)
].head(10)


Original Geolocation Shape:
(1000163, 5)

Duplicate Rows:
261831

Duplicate Percentage:
26.18 %

Sample Duplicate Records:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP


In [10]:

# Step 10: Missing Delivery Date Analysis

missing_delivery = orders[
    orders["order_delivered_customer_date"].isnull()
]

print("Orders with missing delivery dates:")
print(len(missing_delivery))

print("\nOrder Status Distribution:")

print(
    missing_delivery["order_status"].value_counts()
)

print("\nPercentage by Order Status:")

print(
    missing_delivery["order_status"]
    .value_counts(normalize=True) * 100
)


Orders with missing delivery dates:
2965

Order Status Distribution:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

Percentage by Order Status:
order_status
shipped        37.335582
canceled       20.876897
unavailable    20.539629
invoiced       10.590219
processing     10.151771
delivered       0.269815
created         0.168634
approved        0.067454
Name: proportion, dtype: float64


In [11]:

# Find delivered orders with missing delivery dates

delivered_missing = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isnull())
]

print("Inconsistent Delivered Orders:")
print(delivered_missing.shape)

display(delivered_missing)


Inconsistent Delivered Orders:
(8, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [12]:

# Step 12: Geolocation Duplicate Analysis

# Exact duplicate records
exact_duplicates = geolocation[
    geolocation.duplicated(keep=False)
]

print("Exact duplicate records:",
      geolocation.duplicated().sum())

# Check ZIP prefixes with multiple coordinates
zip_analysis = (
    geolocation.groupby("geolocation_zip_code_prefix")
    .agg(
        total_records=("geolocation_zip_code_prefix", "size"),
        unique_latitudes=("geolocation_lat", "nunique"),
        unique_longitudes=("geolocation_lng", "nunique")
    )
    .sort_values("total_records", ascending=False)
)

print("\nZIP prefixes with multiple coordinates:")

display(zip_analysis.head(10))


Exact duplicate records: 261831

ZIP prefixes with multiple coordinates:


,total_records,unique_latitudes,unique_longitudes
geolocation_zip_code_prefix,,,
24220,1146,410,411
24230,1102,327,327
38400,965,746,745
35500,907,726,726
11680,879,726,726
22631,832,141,142
30140,810,379,380
11740,788,666,666
38408,773,600,599


In [13]:

# Step 13: Remove Exact Duplicate Geolocation Records

print("Before Cleaning:", geolocation.shape)

# Remove only completely identical rows
geolocation_cleaned = geolocation.drop_duplicates(
    keep="first"
).copy()

print("After Cleaning:", geolocation_cleaned.shape)

print(
    "Duplicates Remaining:",
    geolocation_cleaned.duplicated().sum()
)

print(
    "Rows Removed:",
    len(geolocation) - len(geolocation_cleaned)
)


Before Cleaning: (1000163, 5)
After Cleaning: (738332, 5)
Duplicates Remaining: 0
Rows Removed: 261831


In [14]:

# Step 14: Convert Order Date Columns

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders_cleaned = orders.copy()

for column in date_columns:
    orders_cleaned[column] = pd.to_datetime(
        orders_cleaned[column],
        errors="coerce"
    )

print("Updated Data Types:")

print(orders_cleaned[date_columns].dtypes)

print("\nMissing Values After Conversion:")

print(orders_cleaned[date_columns].isnull().sum())


Updated Data Types:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Missing Values After Conversion:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [15]:

# Step 15: Flag Inconsistent Delivered Orders

orders_cleaned["delivery_date_issue"] = (
    (orders_cleaned["order_status"] == "delivered") &
    (orders_cleaned["order_delivered_customer_date"].isnull())
)

print(
    "Inconsistent Delivered Orders:",
    orders_cleaned["delivery_date_issue"].sum()
)

display(
    orders_cleaned[
        orders_cleaned["delivery_date_issue"]
    ][
        [
            "order_id",
            "order_status",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delivery_date_issue"
        ]
    ]
)


Inconsistent Delivered Orders: 8


,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_date_issue
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-30 18:12:23,NaT,2017-12-18,True
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-25 08:05:00,NaT,2018-07-16,True
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-03 13:57:00,NaT,2018-07-30,True
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-03 13:57:00,NaT,2018-07-30,True
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-03 09:28:00,NaT,2018-07-24,True
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,NaT,NaT,2017-06-23,True
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-12 14:10:00,NaT,2018-06-26,True
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-07-03 19:26:00,NaT,2018-07-19,True


In [16]:

# Step 16: Detailed Missing Value Analysis

cleaning_datasets = {
    "customers": customers,
    "orders": orders_cleaned,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation_cleaned,
    "category_translation": category_translation
}

missing_report = []

for name, df in cleaning_datasets.items():

    for column in df.columns:

        missing_count = df[column].isnull().sum()

        if missing_count > 0:

            missing_report.append({
                "Dataset": name,
                "Column": column,
                "Missing Values": missing_count,
                "Missing Percentage": round(
                    missing_count / len(df) * 100, 2
                )
            })

missing_df = pd.DataFrame(missing_report)

missing_df = missing_df.sort_values(
    by="Missing Percentage",
    ascending=False
)

display(missing_df)


,Dataset,Column,Missing Values,Missing Percentage
3,reviews,review_comment_title,87656,88.34
4,reviews,review_comment_message,58247,58.70
2,orders,order_delivered_customer_date,2965,2.98
6,products,product_name_lenght,610,1.85
5,products,product_category_name,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
1,orders,order_delivered_carrier_date,1783,1.79
0,orders,order_approved_at,160,0.16
9,products,product_weight_g,2,0.01


In [17]:

# Step 17: Investigate Missing Product Records

product_missing_columns = [
    "product_name_lenght",
    "product_category_name",
    "product_description_lenght",
    "product_photos_qty"
]

print("Missing values pattern:")
display(
    products[product_missing_columns]
    .isnull()
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

print("\nSample missing product records:")
display(
    products[
        products[product_missing_columns].isnull().any(axis=1)
    ].head(10)
)

print("\nMissing order dates by order status:")

for column in [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]:
    print(f"\n{column}")
    display(
        orders_cleaned[
            orders_cleaned[column].isnull()
        ]["order_status"].value_counts()
    )


Missing values pattern:


0    32341
4      610
Name: count, dtype: int64


Sample missing product records:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0



Missing order dates by order status:

order_approved_at


order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64


order_delivered_carrier_date


order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64


order_delivered_customer_date


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [18]:

# Step 18: Missing Order Date Analysis

date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for column in date_columns:

    print(f"\nMissing values in {column}")

    missing_orders = orders_cleaned[
        orders_cleaned[column].isnull()
    ]

    print("Total missing:", len(missing_orders))

    display(
        missing_orders["order_status"]
        .value_counts()
    )

# Product missing records percentage

print("\nProduct Metadata Analysis")

print(
    "Incomplete products:",
    products[product_missing_columns].isnull()
    .all(axis=1).sum()
)

print(
    "Total products:",
    len(products)
)

print(
    "Incomplete percentage:",
    round(
        products[product_missing_columns]
        .isnull().all(axis=1).mean() * 100,
        2
    ), "%"
)



Missing values in order_approved_at
Total missing: 160


order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64


Missing values in order_delivered_carrier_date
Total missing: 1783


order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64


Missing values in order_delivered_customer_date
Total missing: 2965


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


Product Metadata Analysis
Incomplete products: 610
Total products: 32951
Incomplete percentage: 1.85 %


In [19]:

# Step 19: Handle Missing Values Safely

# 1. Create independent cleaned copies

orders_final = orders_cleaned.copy()
products_final = products.copy()
reviews_final = reviews.copy()

# 2. Orders: Create missing date indicators

date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for column in date_columns:

    flag_column = column + "_missing"

    orders_final[flag_column] = (
        orders_final[column].isnull().astype(int)
    )

# 3. Flag delivered orders with missing dates

orders_final["delivery_date_issue"] = (
    (orders_final["order_status"] == "delivered") &
    (
        orders_final["order_delivered_customer_date"].isnull() |
        orders_final["order_delivered_carrier_date"].isnull()
    )
)

# 4. Products: Handle missing categories

products_final["product_category_name"] = (
    products_final["product_category_name"]
    .fillna("unknown")
)

# 5. Create product metadata missing indicators

product_metadata_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

for column in product_metadata_columns:

    products_final[column + "_missing"] = (
        products_final[column].isnull().astype(int)
    )

# 6. Reviews: Preserve missing comments and create flags

review_text_columns = [
    "review_comment_title",
    "review_comment_message"
]

for column in review_text_columns:

    reviews_final[column + "_missing"] = (
        reviews_final[column].isnull().astype(int)
    )

# 7. Display results

print("Orders cleaned:", orders_final.shape)
print("Products cleaned:", products_final.shape)
print("Reviews cleaned:", reviews_final.shape)

print("\nDelivered order date issues:")
print(orders_final["delivery_date_issue"].sum())

print("\nMissing values after treatment:")

print("\nOrders:")
display(orders_final.isnull().sum()[orders_final.isnull().sum() > 0])

print("\nProducts:")
display(products_final.isnull().sum()[products_final.isnull().sum() > 0])

print("\nReviews:")
display(reviews_final.isnull().sum()[reviews_final.isnull().sum() > 0])


Orders cleaned: (99441, 12)
Products cleaned: (32951, 12)
Reviews cleaned: (99224, 9)

Delivered order date issues:
9

Missing values after treatment:

Orders:


order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64


Products:


product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


Reviews:


review_comment_title      87656
review_comment_message    58247
dtype: int64

In [20]:

# Step 20: Handle Missing Numerical Product Values

physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

# Create missing-value indicators
for column in physical_columns:

    products_final[column + "_missing"] = (
        products_final[column].isnull().astype(int)
    )

# Fill missing values using median
for column in physical_columns:

    median_value = products_final[column].median()

    products_final[column] = (
        products_final[column].fillna(median_value)
    )

# Verify results
print("Missing values after numerical imputation:")

print(products_final[physical_columns].isnull().sum())

print("\nUpdated product dataset shape:")
print(products_final.shape)

print("\nProduct sample:")
display(products_final.head())


Missing values after numerical imputation:
product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64

Updated product dataset shape:
(32951, 16)

Product sample:


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_name_lenght_missing,product_description_lenght_missing,product_photos_qty_missing,product_weight_g_missing,product_length_cm_missing,product_height_cm_missing,product_width_cm_missing
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,0,0,0,0,0,0,0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,0,0,0,0,0,0,0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,0,0,0,0,0,0,0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,0,0,0,0,0,0,0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,0,0,0,0,0,0,0


In [21]:

# Step 21: Data Type Validation

print("ORDERS DATA TYPES")
print(orders_cleaned.dtypes)

print("\nPRODUCTS DATA TYPES")
print(products_final.dtypes)

print("\nREVIEWS DATA TYPES")
print(reviews_cleaned.dtypes)

# Check numerical columns
print("\nPRODUCT NUMERICAL SUMMARY")
display(products_final.describe())

# Check datetime columns
print("\nORDER DATETIME COLUMNS")

for column in date_columns:
    print(column, orders_cleaned[column].dtype)


ORDERS DATA TYPES
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
delivery_date_issue                        bool
dtype: object

PRODUCTS DATA TYPES
product_id                                str
product_category_name                     str
product_name_lenght                   float64
product_description_lenght            float64
product_photos_qty                    float64
product_weight_g                      float64
product_length_cm                     float64
product_height_cm                     float64
product_width_cm                      float64
product_name_lenght_missing             int64
product_description_lenght_missing      int64
product_p

NameError: name 'reviews_cleaned' is not defined

In [22]:

# Step 21: Data Type Validation

print("ORDERS DATA TYPES")
print(orders_cleaned.dtypes)

print("\nPRODUCTS DATA TYPES")
print(products_final.dtypes)

print("\nREVIEWS DATA TYPES")
print(reviews_cleaned.dtypes)

# Check numerical columns
print("\nPRODUCT NUMERICAL SUMMARY")
display(products_final.describe())

# Check datetime columns
print("\nORDER DATETIME COLUMNS")

for column in date_columns:
    print(column, orders_cleaned[column].dtype)


ORDERS DATA TYPES
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
delivery_date_issue                        bool
dtype: object

PRODUCTS DATA TYPES
product_id                                str
product_category_name                     str
product_name_lenght                   float64
product_description_lenght            float64
product_photos_qty                    float64
product_weight_g                      float64
product_length_cm                     float64
product_height_cm                     float64
product_width_cm                      float64
product_name_lenght_missing             int64
product_description_lenght_missing      int64
product_p

NameError: name 'reviews_cleaned' is not defined